MODULE **1**

In [24]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/RAG_chatbot_project'
MODELS_DIR = os.path.join(PROJECT_DIR, 'model')
DATACACHE_DIR = os.path.join(PROJECT_DIR, 'data_cache')

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATACACHE_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import os
from datasets import load_dataset, load_from_disk

os.makedirs(DATACACHE_DIR, exist_ok=True)

try:
    print("Trying to load dataset from Google Drive cache...")
    dataset = load_from_disk(DATACACHE_DIR)
    print("Dataset loaded successfully from Google Drive cache!")
except Exception:
    print("Cache not found on Drive. Downloading dataset from Hugging Face...")
    dataset = load_dataset("papluca/language-identification")

    print("Saving dataset to Google Drive cache for future use...")
    dataset.save_to_disk(DATACACHE_DIR)
    print("Dataset saved to Google Drive successfully!")

train_data = dataset["train"]
test_data = dataset["test"]

X_train = train_data["text"]
y_train = train_data["labels"]

X_test = test_data["text"]
y_test = test_data["labels"]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Trying to load dataset from Google Drive cache...
Dataset loaded successfully from Google Drive cache!
Train samples: 70000, Test samples: 10000


In [26]:
!pip install datasets scikit-learn joblib

In [27]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [28]:
# Load dataset from Hugging Face
print("Loading dataset...")
dataset = load_dataset("papluca/language-identification")

train_data = dataset["train"]
test_data = dataset["test"]

# Available columns are ['labels', 'text']
# Extract features and labels using the correct column name 'labels'
X_train = train_data["text"]
y_train = train_data["labels"]

X_test = test_data["text"]
y_test = test_data["labels"]

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Loading dataset...
Train samples: 70000, Test samples: 10000


In [29]:
# Build and train the model using optimized character-level TF-IDF and Logistic Regression
print("Building and training the model...")

from sklearn.pipeline import FeatureUnion

lang_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('char_ft', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), max_features=25000)),
        ('word_ft', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=15000))
    ])),
    ('clf', LogisticRegression(max_iter=1500, C=4.0, class_weight='balanced', n_jobs=-1))
])
# Train the pipeline
lang_pipeline.fit(X_train, y_train)
print("Training completed!")

Building and training the model...
Training completed!


In [30]:
# Evaluate model performance on the test set
print("Evaluating model on test data...")
y_pred = lang_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

Evaluating model on test data...
              precision    recall  f1-score   support

          ar       1.00      1.00      1.00       500
          bg       1.00      1.00      1.00       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       1.00      1.00      1.00       500
          es       0.99      1.00      0.99       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.97      0.98       500
          it       0.99      0.99      0.99       500
          ja       1.00      1.00      1.00       500
          nl       1.00      1.00      1.00       500
          pl       1.00      1.00      1.00       500
          pt       0.99      0.99      0.99       500
          ru       1.00      1.00      1.00       500
          sw       0.96      1.00      0.98       500
          th       1.00      1.00      1.00       500
          tr       0.98      1.00      0.99     

In [31]:
# Save the trained model pipeline for later deployment
model_filename = "language_detector_model.pkl"
joblib.dump(lang_pipeline, model_filename)
print(f"Model saved successfully as {model_filename}!")

Model saved successfully as language_detector_model.pkl!


In [32]:
# Test the model with sample texts in different languages
sample_texts = ["Bonjour tout le monde", "Hello how are you", "السلام عليكم كيف حالك"]
predictions = lang_pipeline.predict(sample_texts)

for text, pred in zip(sample_texts, predictions):
    print(f"Text: '{text}' --> Predicted Language Label: {pred}")

Text: 'Bonjour tout le monde' --> Predicted Language Label: fr
Text: 'Hello how are you' --> Predicted Language Label: en
Text: 'السلام عليكم كيف حالك' --> Predicted Language Label: ar


In [34]:
import os
import joblib

os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "language_detector_model.pkl")

joblib.dump(lang_pipeline, model_path)
print(f"Model saved successfully to Google Drive: {model_path}")

Model saved successfully to Google Drive: /content/drive/MyDrive/RAG_chatbot_project/model/language_detector_model.pkl
